# 🏋️ Gym Reviews & Workout Dataset Scraper
### Targets: JustDial, Sulekha, Practo, Google Places API, Reddit, Google Play Store
**Datasets:**
- `gym_reviews_dataset.csv` — 50,000 gym reviews
- `workout_dataset.csv` — 50,000 workout entries

---
**Author:** Auto-generated  
**Version:** 1.0.0

## 📦 1. Install & Import Dependencies

In [ ]:
# ─── Install required packages ───────────────────────────────────────────────
import subprocess, sys
packages = [
    'requests', 'beautifulsoup4', 'pandas', 'numpy',
    'tqdm', 'lxml', 'cloudscraper', 'fake-useragent',
    'selenium', 'webdriver-manager', 'google-play-scraper',
    'praw'  # Reddit API wrapper
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ All packages installed successfully')

In [ ]:
# ─── Core Imports ─────────────────────────────────────────────────────────────
import os, re, json, time, random, logging, warnings
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin, urlparse, urlencode, quote_plus

import requests
import cloudscraper
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from fake_useragent import UserAgent

# Selenium (for JS-rendered pages)
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s — %(levelname)s — %(message)s')
logger = logging.getLogger(__name__)

# ─── Output Directories ───────────────────────────────────────────────────────
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

print('✅ Imports complete | Output directory ready')

## ⚙️ 2. Global Configuration

In [ ]:
# ─── Configuration ────────────────────────────────────────────────────────────
CONFIG = {
    # Target counts
    'GYM_REVIEWS_TARGET':    50_000,
    'WORKOUT_TARGET':        50_000,

    # Rate-limiting (be polite to servers)
    'MIN_DELAY_SEC':         1.5,
    'MAX_DELAY_SEC':         4.0,
    'MAX_RETRIES':           3,
    'RETRY_BACKOFF':         2,   # exponential backoff multiplier

    # Pagination
    'PAGE_SIZE':             20,   # items per page for most APIs
    'MAX_PAGES_PER_CITY':    50,

    # Checkpoint every N records
    'CHECKPOINT_INTERVAL':   500,

    # ── API Keys (fill in yours) ──────────────────────────────────────────────
    # Google Places API
    'GOOGLE_API_KEY':        os.getenv('GOOGLE_API_KEY', 'YOUR_GOOGLE_PLACES_API_KEY'),

    # Reddit API (https://www.reddit.com/prefs/apps)
    'REDDIT_CLIENT_ID':      os.getenv('REDDIT_CLIENT_ID',     'YOUR_REDDIT_CLIENT_ID'),
    'REDDIT_CLIENT_SECRET':  os.getenv('REDDIT_CLIENT_SECRET', 'YOUR_REDDIT_CLIENT_SECRET'),
    'REDDIT_USER_AGENT':     'FitnessDataBot/1.0',

    # ── Target Indian cities for JustDial / Sulekha ───────────────────────────
    'CITIES': [
        'mumbai', 'delhi', 'bangalore', 'hyderabad', 'chennai',
        'pune', 'kolkata', 'ahmedabad', 'jaipur', 'surat',
        'lucknow', 'kanpur', 'nagpur', 'indore', 'thane',
        'bhopal', 'visakhapatnam', 'pimpri', 'patna', 'vadodara',
        'ghaziabad', 'ludhiana', 'agra', 'nashik', 'faridabad',
        'meerut', 'rajkot', 'kalyan', 'vasai', 'varanasi',
        'chandigarh', 'coimbatore', 'kochi', 'guwahati', 'amritsar'
    ],

    # ── Fitness App IDs (Google Play) ─────────────────────────────────────────
    'FITNESS_APP_IDS': [
        'com.nike.ntc',            # Nike Training Club
        'com.myfitnesspal.android',# MyFitnessPal
        'com.freeletics.freedomen',# Freeletics
        'com.fitify.fitify',       # Fitify
        'com.strongapp.android',   # Strong
        'com.bodybuilding.mobile', # Bodybuilding.com
        'com.sweatco.apps',        # Sweat
        'com.healthifyme.basic',   # HealthifyMe
        'com.cure.fit',            # Cult.fit
        'com.jefit.project',       # JEFIT
        'com.fitbod.fitbod',       # Fitbod
        'com.pumpup.app',          # PumpUp
        'com.runkeeper.android',   # RunKeeper
        'com.strava',              # Strava
        'com.adidas.runtastic',    # Runtastic
        'com.mapmyfitness.android2',# MapMyFitness
        'com.kayla.android',       # SWEAT: Kayla Itsines
        'com.gymshark.training',   # Gymshark Training
    ],
}

print('✅ Configuration loaded')
print(f"   • Gym reviews target  : {CONFIG['GYM_REVIEWS_TARGET']:,}")
print(f"   • Workout data target : {CONFIG['WORKOUT_TARGET']:,}")
print(f"   • Cities targeted     : {len(CONFIG['CITIES'])}")
print(f"   • Fitness apps        : {len(CONFIG['FITNESS_APP_IDS'])}")

## 🔧 3. Utility Helpers

In [ ]:
# ─── User-Agent Pool ──────────────────────────────────────────────────────────
ua = UserAgent()

HEADERS_POOL = [
    {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
        'Accept-Language': 'en-IN,en;q=0.9',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Referer': 'https://www.google.com/',
    },
    {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15',
        'Accept-Language': 'en-US,en;q=0.9',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    },
]

def get_headers():
    """Return a randomised header dict."""
    h = random.choice(HEADERS_POOL).copy()
    h['User-Agent'] = ua.random
    return h


# ─── HTTP Request with Retry ──────────────────────────────────────────────────
def safe_get(url, params=None, timeout=15, use_cloudscraper=False):
    """GET with exponential backoff retries."""
    for attempt in range(1, CONFIG['MAX_RETRIES'] + 1):
        try:
            if use_cloudscraper:
                scraper = cloudscraper.create_scraper()
                resp = scraper.get(url, params=params, timeout=timeout)
            else:
                resp = requests.get(
                    url, params=params,
                    headers=get_headers(), timeout=timeout
                )
            resp.raise_for_status()
            return resp
        except requests.exceptions.HTTPError as e:
            if resp.status_code in (429, 503):
                wait = CONFIG['RETRY_BACKOFF'] ** attempt * 10
                logger.warning(f'Rate-limited ({resp.status_code}). Waiting {wait}s…')
                time.sleep(wait)
            else:
                logger.error(f'HTTP {resp.status_code} on {url}')
                break
        except Exception as e:
            logger.warning(f'Attempt {attempt}/{CONFIG["MAX_RETRIES"]} failed: {e}')
            time.sleep(CONFIG['RETRY_BACKOFF'] ** attempt)
    return None


# ─── Polite Delay ─────────────────────────────────────────────────────────────
def polite_sleep():
    time.sleep(random.uniform(CONFIG['MIN_DELAY_SEC'], CONFIG['MAX_DELAY_SEC']))


# ─── Checkpoint Saver ────────────────────────────────────────────────────────
def save_checkpoint(df: pd.DataFrame, name: str):
    path = CHECKPOINT_DIR / f'{name}_checkpoint_{len(df)}.csv'
    df.to_csv(path, index=False)
    logger.info(f'Checkpoint saved → {path} ({len(df):,} rows)')


# ─── Selenium Setup ───────────────────────────────────────────────────────────
def get_selenium_driver(headless=True):
    """Returns a Chrome WebDriver instance."""
    opts = Options()
    if headless:
        opts.add_argument('--headless=new')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--disable-blink-features=AutomationControlled')
    opts.add_argument(f'user-agent={ua.random}')
    opts.add_experimental_option('excludeSwitches', ['enable-automation'])
    opts.add_experimental_option('useAutomationExtension', False)
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opts
    )
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver


# ─── Text Cleaner ─────────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # remove non-ASCII
    return text


print('✅ Utility helpers ready')

## 🏋️ 4. GYM REVIEWS — Source 1: JustDial Scraper

In [ ]:
# ─── JustDial Gym Listings + Reviews ─────────────────────────────────────────
# JustDial uses a JSON API endpoint; we reverse-engineer the pagination.

JUSTDIAL_BASE = 'https://www.justdial.com'

def scrape_justdial_gyms(city: str, max_pages: int = 50) -> list[dict]:
    """
    Scrape gym listings + reviews from JustDial for one city.
    Uses Selenium to render JS and extract structured data.
    """
    reviews = []
    driver = get_selenium_driver(headless=True)

    try:
        for page in range(1, max_pages + 1):
            url = f'{JUSTDIAL_BASE}/{city}/gyms-fitness-centres/{"page-" + str(page) if page > 1 else ""}'
            driver.get(url)

            # Wait for listings to load
            try:
                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, 'li.cntanr'))
                )
            except Exception:
                logger.warning(f'JustDial: Timeout on {city} page {page}')
                break

            soup = BeautifulSoup(driver.page_source, 'lxml')
            listings = soup.select('li.cntanr')
            if not listings:
                break

            for listing in listings:
                try:
                    # Gym metadata
                    name_el    = listing.select_one('.jdfyname, .fn')
                    rating_el  = listing.select_one('.jdratings, .avgrating')
                    votes_el   = listing.select_one('.votecount, .reviewcount')
                    address_el = listing.select_one('.jdaddr, address')
                    category_el= listing.select_one('.jdtags')
                    phone_el   = listing.select_one('.jdphone, .mobilesv')

                    gym_name    = clean_text(name_el.get_text()) if name_el else 'N/A'
                    rating      = float(rating_el.get_text().strip()) if rating_el else None
                    review_count= clean_text(votes_el.get_text()) if votes_el else '0'
                    address     = clean_text(address_el.get_text()) if address_el else 'N/A'
                    category    = clean_text(category_el.get_text()) if category_el else 'Gym'
                    phone       = clean_text(phone_el.get_text()) if phone_el else 'N/A'

                    # Individual review snippets shown in listing card
                    review_els  = listing.select('.jdreview, .reviewtxt, .review-text')
                    if review_els:
                        for rev_el in review_els:
                            rev_text = clean_text(rev_el.get_text())
                            if rev_text and len(rev_text) > 10:
                                reviews.append({
                                    'source':         'JustDial',
                                    'city':           city.title(),
                                    'gym_name':       gym_name,
                                    'address':        address,
                                    'category':       category,
                                    'phone':          phone,
                                    'gym_rating':     rating,
                                    'total_reviews':  review_count,
                                    'review_text':    rev_text,
                                    'reviewer_name':  'Anonymous',
                                    'review_rating':  rating,
                                    'review_date':    datetime.now().strftime('%Y-%m-%d'),
                                    'scrape_date':    datetime.now().isoformat(),
                                })
                    else:
                        # Store listing-level data even without review text
                        reviews.append({
                            'source':         'JustDial',
                            'city':           city.title(),
                            'gym_name':       gym_name,
                            'address':        address,
                            'category':       category,
                            'phone':          phone,
                            'gym_rating':     rating,
                            'total_reviews':  review_count,
                            'review_text':    '',
                            'reviewer_name':  '',
                            'review_rating':  None,
                            'review_date':    '',
                            'scrape_date':    datetime.now().isoformat(),
                        })

                except Exception as e:
                    logger.debug(f'JustDial listing parse error: {e}')
                    continue

            logger.info(f'JustDial {city} | page {page} | total so far: {len(reviews)}')
            polite_sleep()

    finally:
        driver.quit()

    return reviews


print('✅ JustDial scraper defined')

In [ ]:
# ─── JustDial Detailed Review Page Scraper ───────────────────────────────────
# Visits individual gym pages to scrape full reviews with author & date

def scrape_justdial_gym_reviews(gym_url: str, driver) -> list[dict]:
    """
    Given a JustDial gym detail page URL, scrape all reviews.
    Handles load-more / infinite scroll.
    """
    reviews = []
    try:
        driver.get(gym_url + '#reviews')
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, '.reviewlist, .revlistbox'))
        )

        # Click 'Load More' up to 10 times
        for _ in range(10):
            try:
                load_more = driver.find_element(
                    By.CSS_SELECTOR, '.loadmore-btn, .morereviews'
                )
                driver.execute_script('arguments[0].click();', load_more)
                time.sleep(1.5)
            except Exception:
                break

        soup = BeautifulSoup(driver.page_source, 'lxml')

        # Gym metadata from detail page
        gym_name    = clean_text(soup.select_one('.fn, .bname, h1'))
        gym_addr    = clean_text(soup.select_one('.adr, address'))

        # Review items
        for rev in soup.select('.revlistbox li, .reviewlist li, .rw-list li'):
            author  = rev.select_one('.reviewer-name, .rname')
            date    = rev.select_one('.review-date, .rdate, time')
            rating  = rev.select_one('.star-rating, .rstar')
            text    = rev.select_one('.review-text, .rtxt, .reviewcont')

            review_text = clean_text(text.get_text()) if text else ''
            if not review_text or len(review_text) < 5:
                continue

            # Extract numeric rating from class or text
            star_val = None
            if rating:
                m = re.search(r'(\d(?:\.\d)?)', rating.get('class', [''])[0] + ' ' + rating.get_text())
                if m:
                    star_val = float(m.group(1))

            reviews.append({
                'reviewer_name': clean_text(author.get_text()) if author else 'Anonymous',
                'review_date':   clean_text(date.get_text()) if date else '',
                'review_rating': star_val,
                'review_text':   review_text,
                'gym_name':      gym_name if isinstance(gym_name, str) else '',
                'address':       gym_addr if isinstance(gym_addr, str) else '',
                'gym_url':       gym_url,
                'source':        'JustDial_Detail',
                'scrape_date':   datetime.now().isoformat(),
            })

    except Exception as e:
        logger.warning(f'Detail review scrape failed for {gym_url}: {e}')

    return reviews


print('✅ JustDial detail review scraper defined')

## 🏋️ 5. GYM REVIEWS — Source 2: Sulekha Scraper

In [ ]:
# ─── Sulekha Gym Reviews ──────────────────────────────────────────────────────

SULEKHA_BASE = 'https://www.sulekha.com'

def scrape_sulekha_gyms(city: str, max_pages: int = 40) -> list[dict]:
    """
    Scrape gym listings and reviews from Sulekha for one city.
    Sulekha renders partially server-side; use requests + BS4.
    """
    reviews = []

    for page in range(1, max_pages + 1):
        url = f'{SULEKHA_BASE}/{city}/gyms-and-fitness-centers'
        params = {'page': page}

        resp = safe_get(url, params=params, use_cloudscraper=True)
        if not resp:
            break

        soup = BeautifulSoup(resp.text, 'lxml')

        # Try multiple selector patterns (Sulekha updates their markup)
        listings = (
            soup.select('.biz-listing-compact') or
            soup.select('.listing-item') or
            soup.select('article.spCard') or
            soup.select('.search-result-item')
        )
        if not listings:
            logger.warning(f'Sulekha {city} page {page}: no listings found')
            break

        for listing in listings:
            try:
                name_el   = listing.select_one('h2, .biz-name, .sp-name')
                addr_el   = listing.select_one('.address, .locality')
                rating_el = listing.select_one('.rating-value, .avg-rating')
                count_el  = listing.select_one('.review-count, .ratingcount')
                tags_el   = listing.select_one('.tags, .services')

                gym_name    = clean_text(name_el.get_text()) if name_el else 'N/A'
                address     = clean_text(addr_el.get_text()) if addr_el else 'N/A'
                try:
                    rating  = float(rating_el.get_text().strip()) if rating_el else None
                except ValueError:
                    rating  = None
                count       = clean_text(count_el.get_text()) if count_el else '0'
                tags        = clean_text(tags_el.get_text()) if tags_el else ''

                # Review snippets on listing cards
                rev_els = listing.select('.review-text, .user-review, .review-snippet')
                if rev_els:
                    for rev_el in rev_els:
                        rev_text = clean_text(rev_el.get_text())
                        author_el = rev_el.find_previous('.reviewer, .reviewer-name')
                        date_el   = rev_el.find_next('.review-date, time')

                        if rev_text and len(rev_text) > 10:
                            reviews.append({
                                'source':        'Sulekha',
                                'city':          city.title(),
                                'gym_name':      gym_name,
                                'address':       address,
                                'category':      tags,
                                'gym_rating':    rating,
                                'total_reviews': count,
                                'review_text':   rev_text,
                                'reviewer_name': clean_text(author_el.get_text()) if author_el else 'Anonymous',
                                'review_rating': rating,
                                'review_date':   clean_text(date_el.get_text()) if date_el else '',
                                'scrape_date':   datetime.now().isoformat(),
                            })
                else:
                    reviews.append({
                        'source':        'Sulekha',
                        'city':          city.title(),
                        'gym_name':      gym_name,
                        'address':       address,
                        'category':      tags,
                        'gym_rating':    rating,
                        'total_reviews': count,
                        'review_text':   '',
                        'reviewer_name': '',
                        'review_rating': None,
                        'review_date':   '',
                        'scrape_date':   datetime.now().isoformat(),
                    })

            except Exception as e:
                logger.debug(f'Sulekha parse error: {e}')
                continue

        logger.info(f'Sulekha {city} | page {page} | total: {len(reviews)}')
        polite_sleep()

    return reviews


print('✅ Sulekha scraper defined')

## 🏋️ 6. GYM REVIEWS — Source 3: Google Places API

In [ ]:
# ─── Google Places API — Gym Reviews ─────────────────────────────────────────
# Uses official Places API (Text Search + Place Details)
# Requires API key with Places API enabled

PLACES_BASE = 'https://maps.googleapis.com/maps/api/place'

def google_places_search(query: str, city: str, api_key: str) -> list[dict]:
    """Search for places matching query in a city."""
    url    = f'{PLACES_BASE}/textsearch/json'
    params = {
        'query':  f'{query} in {city} India',
        'key':    api_key,
        'type':   'gym',
        'region': 'IN',
    }
    results = []
    while True:
        resp = safe_get(url, params=params)
        if not resp:
            break
        data = resp.json()
        results.extend(data.get('results', []))
        next_token = data.get('next_page_token')
        if not next_token:
            break
        time.sleep(2)  # Google requires delay before next_page_token is valid
        params = {'key': api_key, 'pagetoken': next_token}
    return results


def google_place_reviews(place_id: str, api_key: str) -> list[dict]:
    """Fetch up to 5 reviews for a place (Places API free tier)."""
    url    = f'{PLACES_BASE}/details/json'
    params = {
        'place_id': place_id,
        'fields':   'name,rating,formatted_address,user_ratings_total,reviews,types,formatted_phone_number',
        'key':      api_key,
        'language': 'en',
    }
    resp = safe_get(url, params=params)
    if not resp:
        return []

    result  = resp.json().get('result', {})
    reviews = result.get('reviews', [])

    parsed = []
    for rev in reviews:
        parsed.append({
            'source':         'Google_Places',
            'city':           '',
            'gym_name':       result.get('name', ''),
            'address':        result.get('formatted_address', ''),
            'category':       ', '.join(result.get('types', [])),
            'phone':          result.get('formatted_phone_number', ''),
            'gym_rating':     result.get('rating'),
            'total_reviews':  result.get('user_ratings_total'),
            'review_text':    clean_text(rev.get('text', '')),
            'reviewer_name':  rev.get('author_name', 'Anonymous'),
            'review_rating':  rev.get('rating'),
            'review_date':    datetime.fromtimestamp(rev.get('time', 0)).strftime('%Y-%m-%d'),
            'place_id':       place_id,
            'scrape_date':    datetime.now().isoformat(),
        })
    return parsed


def scrape_google_places_gyms(
    cities: list[str],
    api_key: str,
    target: int = 50_000
) -> list[dict]:
    """Iterate over cities, search gyms, fetch reviews."""
    all_reviews  = []
    seen_place_ids = set()

    with tqdm(total=target, desc='Google Places Reviews') as pbar:
        for city in cities:
            if len(all_reviews) >= target:
                break
            places = google_places_search('gym fitness center', city, api_key)
            logger.info(f'Google Places: {city} → {len(places)} gyms found')

            for place in places:
                if len(all_reviews) >= target:
                    break
                place_id = place.get('place_id', '')
                if not place_id or place_id in seen_place_ids:
                    continue
                seen_place_ids.add(place_id)

                reviews = google_place_reviews(place_id, api_key)
                for rev in reviews:
                    rev['city'] = city.title()
                all_reviews.extend(reviews)
                pbar.update(len(reviews))
                polite_sleep()

    return all_reviews


print('✅ Google Places scraper defined')

## 🏋️ 7. GYM REVIEWS — Source 4: Practo & UrbanPro Scraper

In [ ]:
# ─── Practo / UrbanPro Fitness Center Reviews ─────────────────────────────────

def scrape_practo_fitness(city: str, max_pages: int = 30) -> list[dict]:
    """
    Scrape fitness trainer / gym reviews from Practo.
    Practo has a public-facing search for fitness trainers.
    """
    reviews = []
    base_url = 'https://www.practo.com'

    for page in range(1, max_pages + 1):
        url = f'{base_url}/{city.lower()}/fitness-trainer'
        params = {'page': page}

        resp = safe_get(url, params=params, use_cloudscraper=True)
        if not resp:
            break

        soup = BeautifulSoup(resp.text, 'lxml')
        cards = soup.select('.listing-doctor-wrap, .u-card, .doctor-listing-card')
        if not cards:
            break

        for card in cards:
            try:
                name_el   = card.select_one('h2, .u-name, .doctor-name')
                spec_el   = card.select_one('.u-specialization, .u-sp')
                rating_el = card.select_one('.ratings-per-doc, .rating')
                count_el  = card.select_one('.count, .u-count')
                loc_el    = card.select_one('.u-location, .clinic-name')

                gym_name = clean_text(name_el.get_text()) if name_el else 'N/A'
                try:
                    rating = float(re.search(r'[\d.]+', rating_el.get_text()).group()) if rating_el else None
                except Exception:
                    rating = None

                rev_snippets = card.select('.patient-review, .review, .feedback-text')
                for rs in rev_snippets:
                    rev_text = clean_text(rs.get_text())
                    if rev_text and len(rev_text) > 10:
                        reviews.append({
                            'source':        'Practo',
                            'city':          city.title(),
                            'gym_name':      gym_name,
                            'address':       clean_text(loc_el.get_text()) if loc_el else '',
                            'category':      clean_text(spec_el.get_text()) if spec_el else 'Fitness Trainer',
                            'gym_rating':    rating,
                            'total_reviews': clean_text(count_el.get_text()) if count_el else '0',
                            'review_text':   rev_text,
                            'reviewer_name': 'Anonymous',
                            'review_rating': rating,
                            'review_date':   '',
                            'scrape_date':   datetime.now().isoformat(),
                        })
                if not rev_snippets:
                    reviews.append({
                        'source':        'Practo',
                        'city':          city.title(),
                        'gym_name':      gym_name,
                        'address':       clean_text(loc_el.get_text()) if loc_el else '',
                        'category':      clean_text(spec_el.get_text()) if spec_el else 'Fitness Trainer',
                        'gym_rating':    rating,
                        'total_reviews': clean_text(count_el.get_text()) if count_el else '0',
                        'review_text':   '',
                        'reviewer_name': '',
                        'review_rating': None,
                        'review_date':   '',
                        'scrape_date':   datetime.now().isoformat(),
                    })
            except Exception as e:
                logger.debug(f'Practo parse error: {e}')

        logger.info(f'Practo {city} | page {page} | total: {len(reviews)}')
        polite_sleep()

    return reviews


print('✅ Practo scraper defined')

## 🏋️ 8. GYM REVIEWS — Source 5: Reddit Fitness Communities (via PRAW)

In [ ]:
# ─── Reddit Fitness Discussions ───────────────────────────────────────────────
import praw

GYM_SUBREDDITS = [
    'india', 'indiafitness', 'fitness', 'gym', 'weightlifting',
    'bodyweightfitness', 'xxfitness', 'GYM', 'WorkOut', 'gymlife',
    'FitIndia', 'mumbai', 'delhi', 'bangalore', 'hyderabad'
]

GYM_KEYWORDS = [
    'gym review', 'gym recommendation', 'best gym', 'gym rating',
    'fitness center', 'cult fit review', 'anytime fitness review',
    'gold gym review', 'fitness first review', 'gym experience',
    'workout place', 'gym membership', 'personal trainer review',
]


def scrape_reddit_gym_reviews(
    client_id: str,
    client_secret: str,
    user_agent: str,
    target: int = 10_000
) -> list[dict]:
    """
    Collect gym review discussions from Reddit using PRAW.
    Each comment in a gym-related thread is treated as a 'review'.
    """
    reddit  = praw.Reddit(
        client_id=client_id,
        client_secret=client_secret,
        user_agent=user_agent,
        read_only=True,
    )
    all_reviews = []
    seen_ids    = set()

    for subreddit_name in tqdm(GYM_SUBREDDITS, desc='Subreddits'):
        if len(all_reviews) >= target:
            break
        try:
            subreddit = reddit.subreddit(subreddit_name)

            for keyword in GYM_KEYWORDS:
                if len(all_reviews) >= target:
                    break
                try:
                    results = subreddit.search(keyword, sort='relevance', limit=100)
                    for submission in results:
                        if submission.id in seen_ids:
                            continue
                        seen_ids.add(submission.id)

                        # Thread body itself
                        if submission.selftext and len(submission.selftext) > 20:
                            all_reviews.append({
                                'source':        'Reddit',
                                'city':          'N/A',
                                'gym_name':      submission.title,
                                'address':       '',
                                'category':      f'r/{subreddit_name}',
                                'gym_rating':    None,
                                'total_reviews': submission.num_comments,
                                'review_text':   clean_text(submission.selftext),
                                'reviewer_name': str(submission.author),
                                'review_rating': None,
                                'review_date':   datetime.fromtimestamp(submission.created_utc).strftime('%Y-%m-%d'),
                                'upvotes':       submission.score,
                                'permalink':     f'https://reddit.com{submission.permalink}',
                                'scrape_date':   datetime.now().isoformat(),
                            })

                        # Top comments
                        submission.comments.replace_more(limit=0)
                        for comment in submission.comments.list()[:20]:
                            if len(all_reviews) >= target:
                                break
                            body = clean_text(comment.body)
                            if body and len(body) > 20 and body != '[deleted]':
                                all_reviews.append({
                                    'source':        'Reddit_Comment',
                                    'city':          'N/A',
                                    'gym_name':      submission.title,
                                    'address':       '',
                                    'category':      f'r/{subreddit_name}',
                                    'gym_rating':    None,
                                    'total_reviews': None,
                                    'review_text':   body,
                                    'reviewer_name': str(comment.author),
                                    'review_rating': None,
                                    'review_date':   datetime.fromtimestamp(comment.created_utc).strftime('%Y-%m-%d'),
                                    'upvotes':       comment.score,
                                    'permalink':     f'https://reddit.com{submission.permalink}',
                                    'scrape_date':   datetime.now().isoformat(),
                                })
                except Exception as e:
                    logger.debug(f'Reddit search error: {e}')
        except Exception as e:
            logger.warning(f'Reddit subreddit r/{subreddit_name} error: {e}')

    return all_reviews


print('✅ Reddit scraper defined')

## 🏃 9. WORKOUT DATASET — Source 1: Google Play Store App Reviews

In [ ]:
# ─── Google Play Store — Fitness App Reviews ─────────────────────────────────
from google_play_scraper import app as gp_app, reviews as gp_reviews, Sort

def scrape_play_store_reviews(
    app_ids: list[str],
    reviews_per_app: int = 3000
) -> list[dict]:
    """
    Scrape reviews from Google Play Store for fitness apps.
    Uses google-play-scraper library.
    """
    all_reviews = []

    for app_id in tqdm(app_ids, desc='Play Store Apps'):
        try:
            # Fetch app metadata
            meta = gp_app(
                app_id,
                lang='en',
                country='in'
            )
            app_name    = meta.get('title', app_id)
            app_rating  = meta.get('score', None)
            app_installs= meta.get('realInstalls', None)
            app_genre   = meta.get('genre', '')

            # Fetch reviews in batches
            result, _ = gp_reviews(
                app_id,
                lang='en',
                country='in',
                sort=Sort.NEWEST,
                count=reviews_per_app,
                filter_score_with=None,  # all ratings
            )

            for rev in result:
                review_text = clean_text(rev.get('content', ''))
                if not review_text or len(review_text) < 5:
                    continue

                all_reviews.append({
                    'source':           'Google_Play',
                    'app_id':           app_id,
                    'app_name':         app_name,
                    'app_genre':        app_genre,
                    'app_overall_rating': app_rating,
                    'app_installs':     app_installs,
                    'reviewer_name':    rev.get('userName', 'Anonymous'),
                    'review_text':      review_text,
                    'review_rating':    rev.get('score'),
                    'review_date':      rev.get('at').strftime('%Y-%m-%d') if rev.get('at') else '',
                    'thumbs_up':        rev.get('thumbsUpCount', 0),
                    'reply_content':    clean_text(rev.get('replyContent', '') or ''),
                    'reply_date':       str(rev.get('repliedAt', '')),
                    'review_id':        rev.get('reviewId', ''),
                    'scrape_date':      datetime.now().isoformat(),
                })

            logger.info(f'Play Store: {app_name} → {len(result)} reviews')
            polite_sleep()

        except Exception as e:
            logger.warning(f'Play Store error for {app_id}: {e}')

    return all_reviews


print('✅ Google Play Store scraper defined')

## 🏃 10. WORKOUT DATASET — Source 2: Reddit Fitness Workouts (via PRAW)

In [ ]:
# ─── Reddit Workout Posts ─────────────────────────────────────────────────────

WORKOUT_SUBREDDITS = [
    'fitness', 'bodyweightfitness', 'weightlifting', 'powerlifting',
    'crossfit', 'running', 'swimming', 'cycling', 'yoga', 'pilates',
    'xxfitness', 'GYM', 'gainit', 'loseit', 'leangains',
    'StartingStrength', 'nSuns', 'PHUL', 'WorkOut', 'indiafitness'
]

WORKOUT_KEYWORDS = [
    'workout routine', 'training program', 'exercise plan', 'workout split',
    'push pull legs', 'upper lower split', 'full body workout',
    'HIIT workout', 'strength training', 'cardio routine',
    'beginner workout', 'advanced workout', 'home workout',
    'chest day', 'leg day', 'back workout', 'shoulder workout',
    'ab workout', 'core workout', 'callisthenics routine',
    'progressive overload', 'RPE training', 'volume training',
]


def scrape_reddit_workouts(
    client_id: str,
    client_secret: str,
    user_agent: str,
    target: int = 15_000
) -> list[dict]:
    """
    Collect workout posts and comments from Reddit fitness communities.
    """
    reddit  = praw.Reddit(
        client_id=client_id,
        client_secret=client_secret,
        user_agent=user_agent,
        read_only=True,
    )
    all_posts = []
    seen_ids  = set()

    for sr_name in tqdm(WORKOUT_SUBREDDITS, desc='Workout Subreddits'):
        if len(all_posts) >= target:
            break
        try:
            sr = reddit.subreddit(sr_name)

            # Hot & top posts for general routines
            for submission in list(sr.hot(limit=200)) + list(sr.top('year', limit=200)):
                if len(all_posts) >= target:
                    break
                if submission.id in seen_ids:
                    continue
                seen_ids.add(submission.id)

                body = clean_text(submission.selftext)
                if body and len(body) > 30:
                    all_posts.append({
                        'source':       'Reddit_Workout',
                        'subreddit':    sr_name,
                        'post_id':      submission.id,
                        'title':        submission.title,
                        'workout_text': body,
                        'author':       str(submission.author),
                        'upvotes':      submission.score,
                        'num_comments': submission.num_comments,
                        'post_type':    'submission',
                        'date':         datetime.fromtimestamp(submission.created_utc).strftime('%Y-%m-%d'),
                        'permalink':    f'https://reddit.com{submission.permalink}',
                        'flair':        submission.link_flair_text or '',
                        'scrape_date':  datetime.now().isoformat(),
                    })

                # Comments with detailed workout content
                submission.comments.replace_more(limit=0)
                for comment in submission.comments.list()[:15]:
                    if len(all_posts) >= target:
                        break
                    ctext = clean_text(comment.body)
                    if ctext and len(ctext) > 50 and ctext != '[deleted]':
                        all_posts.append({
                            'source':       'Reddit_Workout_Comment',
                            'subreddit':    sr_name,
                            'post_id':      submission.id,
                            'title':        submission.title,
                            'workout_text': ctext,
                            'author':       str(comment.author),
                            'upvotes':      comment.score,
                            'num_comments': None,
                            'post_type':    'comment',
                            'date':         datetime.fromtimestamp(comment.created_utc).strftime('%Y-%m-%d'),
                            'permalink':    f'https://reddit.com{submission.permalink}',
                            'flair':        '',
                            'scrape_date':  datetime.now().isoformat(),
                        })

            # Keyword search
            for kw in WORKOUT_KEYWORDS[:5]:  # limit per subreddit
                if len(all_posts) >= target:
                    break
                for sub in sr.search(kw, sort='top', limit=50):
                    if sub.id in seen_ids:
                        continue
                    seen_ids.add(sub.id)
                    body = clean_text(sub.selftext)
                    if body and len(body) > 30:
                        all_posts.append({
                            'source':       'Reddit_Workout_Search',
                            'subreddit':    sr_name,
                            'post_id':      sub.id,
                            'title':        sub.title,
                            'workout_text': body,
                            'author':       str(sub.author),
                            'upvotes':      sub.score,
                            'num_comments': sub.num_comments,
                            'post_type':    'search_result',
                            'date':         datetime.fromtimestamp(sub.created_utc).strftime('%Y-%m-%d'),
                            'permalink':    f'https://reddit.com{sub.permalink}',
                            'flair':        sub.link_flair_text or '',
                            'scrape_date':  datetime.now().isoformat(),
                        })

        except Exception as e:
            logger.warning(f'Reddit workout error r/{sr_name}: {e}')

    return all_posts


print('✅ Reddit workout scraper defined')

## 🏃 11. WORKOUT DATASET — Source 3: JustDial / Sulekha Workout / PT Data

In [ ]:
# ─── JustDial Fitness / Yoga / CrossFit Classes Listings ─────────────────────

WORKOUT_CATEGORIES = [
    'yoga-classes', 'zumba-classes', 'crossfit-centres',
    'aerobics-classes', 'pilates-classes', 'martial-arts-classes',
    'personal-fitness-trainers', 'swimming-classes', 'dance-fitness-classes',
    'cycling-classes', 'functional-fitness-centers', 'bootcamp-fitness',
]


def scrape_justdial_workout_classes(city: str, max_pages: int = 30) -> list[dict]:
    """
    Scrape workout class listings and reviews from JustDial.
    Covers yoga, Zumba, CrossFit, aerobics, etc.
    """
    all_data = []
    driver   = get_selenium_driver(headless=True)

    try:
        for category in WORKOUT_CATEGORIES:
            for page in range(1, max_pages + 1):
                url = f'{JUSTDIAL_BASE}/{city}/{category}'
                if page > 1:
                    url += f'/page-{page}'

                driver.get(url)
                try:
                    WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, 'li.cntanr'))
                    )
                except Exception:
                    break

                soup     = BeautifulSoup(driver.page_source, 'lxml')
                listings = soup.select('li.cntanr')
                if not listings:
                    break

                for listing in listings:
                    try:
                        name_el  = listing.select_one('.jdfyname, .fn')
                        rat_el   = listing.select_one('.jdratings, .avgrating')
                        addr_el  = listing.select_one('.jdaddr, address')
                        tags_el  = listing.select_one('.jdtags')
                        price_el = listing.select_one('.jdprice, .price')
                        rev_els  = listing.select('.jdreview, .reviewtxt')

                        class_name = clean_text(name_el.get_text()) if name_el else 'N/A'
                        address    = clean_text(addr_el.get_text()) if addr_el else 'N/A'
                        tags       = clean_text(tags_el.get_text()) if tags_el else ''
                        price      = clean_text(price_el.get_text()) if price_el else ''
                        try:
                            rating = float(rat_el.get_text().strip()) if rat_el else None
                        except ValueError:
                            rating = None

                        entry_base = {
                            'source':         'JustDial_Workout',
                            'city':           city.title(),
                            'class_name':     class_name,
                            'workout_type':   category.replace('-', ' ').title(),
                            'address':        address,
                            'tags':           tags,
                            'price_info':     price,
                            'class_rating':   rating,
                            'scrape_date':    datetime.now().isoformat(),
                        }

                        if rev_els:
                            for rev_el in rev_els:
                                rev_text = clean_text(rev_el.get_text())
                                if rev_text and len(rev_text) > 10:
                                    entry = entry_base.copy()
                                    entry['review_text']   = rev_text
                                    entry['reviewer_name'] = 'Anonymous'
                                    entry['review_date']   = ''
                                    all_data.append(entry)
                        else:
                            entry = entry_base.copy()
                            entry['review_text']   = ''
                            entry['reviewer_name'] = ''
                            entry['review_date']   = ''
                            all_data.append(entry)

                    except Exception as e:
                        logger.debug(f'JustDial workout parse error: {e}')

                logger.info(f'JustDial Workout {city}/{category} | page {page} | total: {len(all_data)}')
                polite_sleep()

    finally:
        driver.quit()

    return all_data


print('✅ JustDial workout class scraper defined')

## 🏃 12. WORKOUT DATASET — Source 4: HealthifyMe & Cult.fit Public Data

In [ ]:
# ─── HealthifyMe Blog / Workout Articles ──────────────────────────────────────

def scrape_healthifyme_workouts(max_pages: int = 100) -> list[dict]:
    """
    Scrape workout articles and plans from HealthifyMe blog.
    Each article is treated as a workout 'entry'.
    """
    articles = []
    base_url = 'https://healthifyme.com/blog/category/exercise'

    for page in range(1, max_pages + 1):
        url  = f'{base_url}/page/{page}/' if page > 1 else base_url
        resp = safe_get(url, use_cloudscraper=True)
        if not resp:
            break

        soup = BeautifulSoup(resp.text, 'lxml')
        posts = soup.select('article.post, .blog-card, .entry')
        if not posts:
            break

        for post in posts:
            try:
                title_el  = post.select_one('h2, h3, .entry-title')
                excerpt_el= post.select_one('.excerpt, .entry-summary, p')
                date_el   = post.select_one('time, .date')
                tag_els   = post.select('.tag, .category, .label')
                link_el   = post.select_one('a')

                title   = clean_text(title_el.get_text()) if title_el else ''
                excerpt = clean_text(excerpt_el.get_text()) if excerpt_el else ''
                date    = date_el.get('datetime', clean_text(date_el.get_text())) if date_el else ''
                tags    = [clean_text(t.get_text()) for t in tag_els]
                url_    = urljoin('https://healthifyme.com', link_el['href']) if link_el and link_el.get('href') else ''

                if title and (excerpt or url_):
                    articles.append({
                        'source':        'HealthifyMe_Blog',
                        'title':         title,
                        'workout_type':  ', '.join(tags) or 'Exercise',
                        'workout_text':  excerpt,
                        'date':          str(date),
                        'url':           url_,
                        'scrape_date':   datetime.now().isoformat(),
                    })
            except Exception as e:
                logger.debug(f'HealthifyMe parse error: {e}')

        logger.info(f'HealthifyMe page {page} | articles: {len(articles)}')
        polite_sleep()

    return articles


# ─── Cult.fit Class Listings ──────────────────────────────────────────────────

def scrape_cultfit_classes(cities: list[str]) -> list[dict]:
    """
    Scrape publicly-listed workout classes from Cult.fit.
    """
    classes  = []
    base_url = 'https://cult.fit'

    for city in cities:
        url  = f'{base_url}/cult-sport/fitness-classes/{city.lower()}'
        resp = safe_get(url, use_cloudscraper=True)
        if not resp:
            continue

        soup = BeautifulSoup(resp.text, 'lxml')

        # Try multiple patterns
        items = (
            soup.select('.class-card, .activity-card') or
            soup.select('[class*="ClassCard"], [class*="ActivityCard"]') or
            soup.select('article')
        )

        for item in items:
            try:
                name_el  = item.select_one('h2, h3, [class*="title"], [class*="name"]')
                desc_el  = item.select_one('p, [class*="desc"], [class*="about"]')
                type_el  = item.select_one('[class*="category"], [class*="type"], .tag')
                dur_el   = item.select_one('[class*="duration"], [class*="time"]')
                cal_el   = item.select_one('[class*="calorie"], [class*="burn"]')
                diff_el  = item.select_one('[class*="difficulty"], [class*="level"]')

                name = clean_text(name_el.get_text()) if name_el else ''
                if not name:
                    continue

                classes.append({
                    'source':        'CultFit',
                    'city':          city.title(),
                    'class_name':    name,
                    'workout_type':  clean_text(type_el.get_text()) if type_el else 'Fitness',
                    'workout_text':  clean_text(desc_el.get_text()) if desc_el else '',
                    'duration_mins': clean_text(dur_el.get_text()) if dur_el else '',
                    'calories_burn': clean_text(cal_el.get_text()) if cal_el else '',
                    'difficulty':    clean_text(diff_el.get_text()) if diff_el else '',
                    'date':          datetime.now().strftime('%Y-%m-%d'),
                    'scrape_date':   datetime.now().isoformat(),
                })
            except Exception as e:
                logger.debug(f'Cult.fit parse error: {e}')

        logger.info(f'Cult.fit {city} → {len(classes)} classes')
        polite_sleep()

    return classes


print('✅ HealthifyMe & Cult.fit scrapers defined')

## 🚀 13. RUN ALL SCRAPERS — Gym Reviews Dataset

In [ ]:
# ─── MASTER RUNNER — Gym Reviews ─────────────────────────────────────────────
# NOTE: Replace API keys in CONFIG before running.
# Each scraper can be run independently; results are combined.

GYM_REVIEWS_TARGET = CONFIG['GYM_REVIEWS_TARGET']   # 50,000
all_gym_reviews    = []

print(f'🏋️  Starting Gym Reviews collection | Target: {GYM_REVIEWS_TARGET:,}')
print('=' * 60)

# ── SOURCE 1: JustDial ────────────────────────────────────────────
print('\n[1/5] Scraping JustDial...')
for city in tqdm(CONFIG['CITIES'], desc='JustDial cities'):
    if len(all_gym_reviews) >= GYM_REVIEWS_TARGET:
        break
    try:
        reviews = scrape_justdial_gyms(city, max_pages=CONFIG['MAX_PAGES_PER_CITY'])
        all_gym_reviews.extend(reviews)
        print(f'  ✓ {city.title()}: +{len(reviews)} | total: {len(all_gym_reviews):,}')

        # Checkpoint
        if len(all_gym_reviews) % CONFIG['CHECKPOINT_INTERVAL'] < len(reviews):
            df_cp = pd.DataFrame(all_gym_reviews)
            save_checkpoint(df_cp, 'gym_reviews')
    except Exception as e:
        logger.error(f'JustDial {city} failed: {e}')

print(f'  → JustDial total: {len(all_gym_reviews):,}')


# ── SOURCE 2: Sulekha ─────────────────────────────────────────────
print('\n[2/5] Scraping Sulekha...')
for city in tqdm(CONFIG['CITIES'], desc='Sulekha cities'):
    if len(all_gym_reviews) >= GYM_REVIEWS_TARGET:
        break
    try:
        reviews = scrape_sulekha_gyms(city, max_pages=40)
        all_gym_reviews.extend(reviews)
        print(f'  ✓ {city.title()}: +{len(reviews)} | total: {len(all_gym_reviews):,}')
    except Exception as e:
        logger.error(f'Sulekha {city} failed: {e}')

print(f'  → Sulekha total: {len(all_gym_reviews):,}')


# ── SOURCE 3: Google Places (requires API key) ────────────────────
print('\n[3/5] Scraping Google Places API...')
if CONFIG['GOOGLE_API_KEY'] not in ('YOUR_GOOGLE_PLACES_API_KEY', ''):
    try:
        gp_reviews = scrape_google_places_gyms(
            cities=CONFIG['CITIES'],
            api_key=CONFIG['GOOGLE_API_KEY'],
            target=max(0, GYM_REVIEWS_TARGET - len(all_gym_reviews))
        )
        all_gym_reviews.extend(gp_reviews)
        print(f'  → Google Places: +{len(gp_reviews)} | total: {len(all_gym_reviews):,}')
    except Exception as e:
        logger.error(f'Google Places failed: {e}')
else:
    print('  ⚠ Google Places API key not set — skipping')


# ── SOURCE 4: Practo ──────────────────────────────────────────────
print('\n[4/5] Scraping Practo...')
for city in tqdm(CONFIG['CITIES'][:15], desc='Practo cities'):
    if len(all_gym_reviews) >= GYM_REVIEWS_TARGET:
        break
    try:
        reviews = scrape_practo_fitness(city, max_pages=30)
        all_gym_reviews.extend(reviews)
        print(f'  ✓ {city.title()}: +{len(reviews)} | total: {len(all_gym_reviews):,}')
    except Exception as e:
        logger.error(f'Practo {city} failed: {e}')

print(f'  → Practo total: {len(all_gym_reviews):,}')


# ── SOURCE 5: Reddit ──────────────────────────────────────────────
print('\n[5/5] Scraping Reddit...')
if CONFIG['REDDIT_CLIENT_ID'] not in ('YOUR_REDDIT_CLIENT_ID', ''):
    try:
        reddit_revs = scrape_reddit_gym_reviews(
            client_id=CONFIG['REDDIT_CLIENT_ID'],
            client_secret=CONFIG['REDDIT_CLIENT_SECRET'],
            user_agent=CONFIG['REDDIT_USER_AGENT'],
            target=max(0, GYM_REVIEWS_TARGET - len(all_gym_reviews))
        )
        all_gym_reviews.extend(reddit_revs)
        print(f'  → Reddit: +{len(reddit_revs)} | total: {len(all_gym_reviews):,}')
    except Exception as e:
        logger.error(f'Reddit gym reviews failed: {e}')
else:
    print('  ⚠ Reddit API credentials not set — skipping')


print(f'\n✅ Gym reviews collected: {len(all_gym_reviews):,}')

## 🚀 14. RUN ALL SCRAPERS — Workout Dataset

In [ ]:
# ─── MASTER RUNNER — Workout Dataset ─────────────────────────────────────────

WORKOUT_TARGET  = CONFIG['WORKOUT_TARGET']   # 50,000
all_workouts    = []

print(f'🏃 Starting Workout Data collection | Target: {WORKOUT_TARGET:,}')
print('=' * 60)


# ── SOURCE 1: Google Play Store ───────────────────────────────────
print('\n[1/4] Scraping Google Play Store...')
try:
    needed           = max(0, WORKOUT_TARGET - len(all_workouts))
    reviews_per_app  = max(100, needed // len(CONFIG['FITNESS_APP_IDS']))
    play_reviews     = scrape_play_store_reviews(
        app_ids=CONFIG['FITNESS_APP_IDS'],
        reviews_per_app=reviews_per_app
    )
    all_workouts.extend(play_reviews)
    print(f'  → Play Store: +{len(play_reviews)} | total: {len(all_workouts):,}')
except Exception as e:
    logger.error(f'Play Store scraping failed: {e}')


# ── SOURCE 2: Reddit Workout Posts ───────────────────────────────
print('\n[2/4] Scraping Reddit Workout Posts...')
if CONFIG['REDDIT_CLIENT_ID'] not in ('YOUR_REDDIT_CLIENT_ID', ''):
    try:
        reddit_workouts = scrape_reddit_workouts(
            client_id=CONFIG['REDDIT_CLIENT_ID'],
            client_secret=CONFIG['REDDIT_CLIENT_SECRET'],
            user_agent=CONFIG['REDDIT_USER_AGENT'],
            target=max(0, WORKOUT_TARGET - len(all_workouts))
        )
        all_workouts.extend(reddit_workouts)
        print(f'  → Reddit workouts: +{len(reddit_workouts)} | total: {len(all_workouts):,}')
    except Exception as e:
        logger.error(f'Reddit workout failed: {e}')
else:
    print('  ⚠ Reddit API credentials not set — skipping')


# ── SOURCE 3: JustDial Workout Classes ───────────────────────────
print('\n[3/4] Scraping JustDial Workout Classes...')
for city in tqdm(CONFIG['CITIES'], desc='JD Workout cities'):
    if len(all_workouts) >= WORKOUT_TARGET:
        break
    try:
        data = scrape_justdial_workout_classes(city, max_pages=20)
        all_workouts.extend(data)
        print(f'  ✓ {city.title()}: +{len(data)} | total: {len(all_workouts):,}')
        if len(all_workouts) % CONFIG['CHECKPOINT_INTERVAL'] < len(data):
            save_checkpoint(pd.DataFrame(all_workouts), 'workouts')
    except Exception as e:
        logger.error(f'JD Workout {city} failed: {e}')


# ── SOURCE 4: HealthifyMe & Cult.fit ─────────────────────────────
print('\n[4/4] Scraping HealthifyMe & Cult.fit...')
try:
    hm_articles = scrape_healthifyme_workouts(max_pages=100)
    all_workouts.extend(hm_articles)
    print(f'  → HealthifyMe: +{len(hm_articles)}')
except Exception as e:
    logger.error(f'HealthifyMe failed: {e}')

try:
    cf_classes = scrape_cultfit_classes(CONFIG['CITIES'][:20])
    all_workouts.extend(cf_classes)
    print(f'  → Cult.fit: +{len(cf_classes)}')
except Exception as e:
    logger.error(f'Cult.fit failed: {e}')


print(f'\n✅ Workout entries collected: {len(all_workouts):,}')

## 🧹 15. Data Cleaning & Standardisation

In [ ]:
# ─── GYM REVIEWS: Clean & Standardise ────────────────────────────────────────

GYM_COLUMNS = [
    'source', 'city', 'gym_name', 'address', 'category', 'phone',
    'gym_rating', 'total_reviews', 'reviewer_name',
    'review_rating', 'review_text', 'review_date',
    'upvotes', 'place_id', 'permalink', 'scrape_date'
]

def clean_gym_reviews(records: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(records)

    # Ensure all columns exist
    for col in GYM_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan
    df = df[GYM_COLUMNS]

    # Deduplicate on review_text + gym_name
    before = len(df)
    df.drop_duplicates(subset=['review_text', 'gym_name'], keep='first', inplace=True)
    print(f'  Dedup: {before:,} → {len(df):,} (removed {before - len(df):,})')

    # Clean text fields
    for col in ['gym_name', 'address', 'review_text', 'reviewer_name', 'category']:
        df[col] = df[col].astype(str).apply(clean_text)
        df[col] = df[col].replace({'nan': '', 'N/A': '', 'None': ''})

    # Numeric
    df['gym_rating']    = pd.to_numeric(df['gym_rating'],    errors='coerce').clip(0, 5)
    df['review_rating'] = pd.to_numeric(df['review_rating'], errors='coerce').clip(0, 5)

    # Standardise dates
    df['review_date']  = pd.to_datetime(df['review_date'],  errors='coerce').dt.strftime('%Y-%m-%d')
    df['scrape_date']  = pd.to_datetime(df['scrape_date'],  errors='coerce').dt.strftime('%Y-%m-%d')

    # Remove rows with no useful content
    df = df[
        (df['gym_name'].str.len() > 1) |
        (df['review_text'].str.len() > 5)
    ].reset_index(drop=True)

    # Derived columns
    df['review_length'] = df['review_text'].str.len()
    df['has_review']    = (df['review_text'].str.len() > 5).astype(int)

    # Sentiment placeholder (can run VADER/TextBlob separately)
    df['sentiment_label'] = np.where(
        df['review_rating'] >= 4, 'Positive',
        np.where(df['review_rating'] <= 2, 'Negative', 'Neutral')
    )

    return df


# ─── WORKOUT: Clean & Standardise ────────────────────────────────────────────

WORKOUT_COLUMNS = [
    'source', 'app_name', 'app_id', 'app_genre', 'app_overall_rating',
    'app_installs', 'subreddit', 'city', 'class_name', 'workout_type',
    'workout_text', 'review_text', 'title', 'reviewer_name', 'author',
    'review_rating', 'review_date', 'date', 'thumbs_up', 'upvotes',
    'duration_mins', 'calories_burn', 'difficulty', 'price_info',
    'tags', 'url', 'permalink', 'review_id', 'scrape_date'
]

def clean_workout_data(records: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(records)

    for col in WORKOUT_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan
    df = df[WORKOUT_COLUMNS]

    # Unified text column: prefer workout_text, fall back to review_text
    df['content'] = df['workout_text'].fillna('').astype(str)
    mask = df['content'].str.len() < 5
    df.loc[mask, 'content'] = df.loc[mask, 'review_text'].fillna('').astype(str)

    # Unified name column: prefer class_name, fall back to app_name / title
    df['item_name'] = (
        df['class_name'].fillna('') + df['app_name'].fillna('') + df['title'].fillna('')
    ).str.strip()

    # Unified date
    df['entry_date'] = pd.to_datetime(
        df['date'].fillna(df['review_date']), errors='coerce'
    ).dt.strftime('%Y-%m-%d')

    # Deduplicate
    before = len(df)
    df.drop_duplicates(subset=['content', 'item_name'], keep='first', inplace=True)
    print(f'  Dedup: {before:,} → {len(df):,} (removed {before - len(df):,})')

    # Clean text
    for col in ['content', 'item_name', 'workout_type']:
        df[col] = df[col].astype(str).apply(clean_text).replace({'nan': '', 'None': ''})

    df = df[df['content'].str.len() > 5].reset_index(drop=True)

    # Derived
    df['content_length'] = df['content'].str.len()
    df['review_rating']  = pd.to_numeric(df['review_rating'], errors='coerce').clip(1, 5)

    return df


print('✅ Cleaning functions defined')

In [ ]:
# ─── Apply Cleaning ───────────────────────────────────────────────────────────
print('Cleaning gym reviews dataset...')
df_gym = clean_gym_reviews(all_gym_reviews)
print(f'  Final gym reviews: {len(df_gym):,}')

print('\nCleaning workout dataset...')
df_workout = clean_workout_data(all_workouts)
print(f'  Final workout entries: {len(df_workout):,}')

## 📊 16. Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'figure.facecolor': '#1a1a2e', 'text.color': 'white',
                     'axes.facecolor': '#16213e', 'axes.edgecolor': '#e94560',
                     'xtick.color': 'white', 'ytick.color': 'white',
                     'axes.labelcolor': 'white', 'axes.titlecolor': 'white'})

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('📊 Gym Reviews & Workout Dataset — EDA', fontsize=16, color='white')

# ── 1. Source distribution — Gym Reviews ─────────────────────────
ax = axes[0, 0]
src_counts = df_gym['source'].value_counts()
ax.bar(src_counts.index, src_counts.values, color='#e94560')
ax.set_title('Gym Reviews by Source')
ax.set_xlabel('Source')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=30)

# ── 2. Rating distribution — Gym Reviews ─────────────────────────
ax = axes[0, 1]
df_gym['review_rating'].dropna().hist(bins=10, ax=ax, color='#0f3460', edgecolor='#e94560')
ax.set_title('Review Rating Distribution')
ax.set_xlabel('Rating')
ax.set_ylabel('Count')

# ── 3. Top cities ─────────────────────────────────────────────────
ax = axes[0, 2]
city_counts = df_gym['city'].value_counts().head(15)
ax.barh(city_counts.index[::-1], city_counts.values[::-1], color='#533483')
ax.set_title('Top 15 Cities — Gym Reviews')
ax.set_xlabel('Reviews')

# ── 4. Workout source distribution ───────────────────────────────
ax = axes[1, 0]
ws_counts = df_workout['source'].value_counts()
ax.bar(ws_counts.index, ws_counts.values, color='#e94560')
ax.set_title('Workout Data by Source')
ax.set_xlabel('Source')
ax.tick_params(axis='x', rotation=30)

# ── 5. Workout type distribution ─────────────────────────────────
ax = axes[1, 1]
wt_counts = df_workout['workout_type'].value_counts().head(12)
ax.barh(wt_counts.index[::-1], wt_counts.values[::-1], color='#0f3460')
ax.set_title('Top Workout Types')
ax.set_xlabel('Count')

# ── 6. Sentiment distribution ─────────────────────────────────────
ax = axes[1, 2]
sent_counts = df_gym['sentiment_label'].value_counts()
colors = {'Positive': '#4ade80', 'Neutral': '#facc15', 'Negative': '#f87171'}
ax.pie(
    sent_counts.values,
    labels=sent_counts.index,
    colors=[colors.get(l, '#888') for l in sent_counts.index],
    autopct='%1.1f%%',
    textprops={'color': 'white'}
)
ax.set_title('Gym Review Sentiment')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved')

In [ ]:
# ─── Summary Statistics ───────────────────────────────────────────────────────
print('══════════════════════════════════════════════')
print('         GYM REVIEWS DATASET SUMMARY          ')
print('══════════════════════════════════════════════')
print(f'Total records      : {len(df_gym):,}')
print(f'Records w/ review  : {df_gym["has_review"].sum():,}')
print(f'Unique gyms        : {df_gym["gym_name"].nunique():,}')
print(f'Cities covered     : {df_gym["city"].nunique():,}')
print(f'Sources            : {df_gym["source"].nunique()}')
print(f'Avg gym rating     : {df_gym["gym_rating"].mean():.2f}')
print(f'Avg review length  : {df_gym["review_length"].mean():.0f} chars')
print()
print('── Source Breakdown ──────────────────────────')
print(df_gym['source'].value_counts().to_string())

print()
print('══════════════════════════════════════════════')
print('         WORKOUT DATASET SUMMARY              ')
print('══════════════════════════════════════════════')
print(f'Total records      : {len(df_workout):,}')
print(f'Unique items       : {df_workout["item_name"].nunique():,}')
print(f'Sources            : {df_workout["source"].nunique()}')
print(f'Avg content length : {df_workout["content_length"].mean():.0f} chars')
print()
print('── Source Breakdown ──────────────────────────')
print(df_workout['source'].value_counts().to_string())

## 💾 17. Export to CSV

In [ ]:
# ─── Save Final CSVs ──────────────────────────────────────────────────────────

GYM_CSV     = OUTPUT_DIR / 'gym_reviews_dataset.csv'
WORKOUT_CSV = OUTPUT_DIR / 'workout_dataset.csv'

# Save Gym Reviews
df_gym.to_csv(GYM_CSV, index=False, encoding='utf-8-sig')
gym_size = GYM_CSV.stat().st_size / 1024 / 1024
print(f'✅ Gym Reviews saved → {GYM_CSV}')
print(f'   Rows: {len(df_gym):,} | Size: {gym_size:.2f} MB')

# Save Workout Dataset
df_workout.to_csv(WORKOUT_CSV, index=False, encoding='utf-8-sig')
workout_size = WORKOUT_CSV.stat().st_size / 1024 / 1024
print(f'\n✅ Workout Dataset saved → {WORKOUT_CSV}')
print(f'   Rows: {len(df_workout):,} | Size: {workout_size:.2f} MB')

# Save column schemas
schema_gym     = pd.DataFrame({'column': df_gym.columns, 'dtype': df_gym.dtypes.values, 'non_null': df_gym.notna().sum().values})
schema_workout = pd.DataFrame({'column': df_workout.columns, 'dtype': df_workout.dtypes.values, 'non_null': df_workout.notna().sum().values})
schema_gym.to_csv(OUTPUT_DIR / 'schema_gym_reviews.csv', index=False)
schema_workout.to_csv(OUTPUT_DIR / 'schema_workout.csv', index=False)

print('\n✅ Schema files saved')
print('\n📂 Output directory contents:')
for f in sorted(OUTPUT_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print(f'   {f.name:<40} {size:.1f} KB')

## 🔍 18. Quick Validation & Samples

In [ ]:
# ─── Sample rows from each dataset ───────────────────────────────────────────
print('── GYM REVIEWS — Sample (5 rows with review text) ──────────────')
sample_gym = df_gym[df_gym['review_text'].str.len() > 20].head(5)
for _, row in sample_gym.iterrows():
    print(f"  [{row['source']}] {row['gym_name']} | ⭐ {row['review_rating']}")
    print(f"  {row['review_text'][:120]}...")
    print()

print('── WORKOUT DATA — Sample (5 rows) ──────────────────────────────')
sample_wk = df_workout[df_workout['content'].str.len() > 30].head(5)
for _, row in sample_wk.iterrows():
    print(f"  [{row['source']}] {row['item_name']} | {row['workout_type']}")
    print(f"  {row['content'][:120]}...")
    print()

In [ ]:
# ─── Data Quality Report ──────────────────────────────────────────────────────
print('═══════════════════════════ DATA QUALITY REPORT ═══════════════════════════')

for name, df in [('Gym Reviews', df_gym), ('Workout Dataset', df_workout)]:
    print(f'\n── {name} ──')
    null_pct = (df.isnull().mean() * 100).round(1)
    print(null_pct[null_pct > 0].to_string())
    print(f'  Total rows : {len(df):,}')
    print(f'  Columns    : {len(df.columns)}')

print('\n✅ All done! Datasets are ready.')

---
## 📋 Dataset Schema Reference

### `gym_reviews_dataset.csv`
| Column | Type | Description |
|---|---|---|
| source | str | Data source (JustDial/Sulekha/Google_Places/Reddit) |
| city | str | Indian city name |
| gym_name | str | Name of the gym/fitness centre |
| address | str | Physical address |
| category | str | Type (gym/yoga/crossfit etc.) |
| phone | str | Contact number |
| gym_rating | float | Overall rating (0–5) |
| total_reviews | str | Total review count |
| reviewer_name | str | Name of reviewer |
| review_rating | float | Rating given by reviewer (0–5) |
| review_text | str | Review content |
| review_date | date | Date of review |
| upvotes | int | Reddit upvotes (if applicable) |
| place_id | str | Google Place ID (if applicable) |
| review_length | int | Length of review text |
| has_review | int | Binary: 1 if text present |
| sentiment_label | str | Positive / Neutral / Negative |

### `workout_dataset.csv`
| Column | Type | Description |
|---|---|---|
| source | str | Data source |
| app_name | str | Fitness app name |
| app_genre | str | App category |
| workout_type | str | Type of workout |
| content | str | Unified workout/review text |
| item_name | str | Unified name field |
| review_rating | float | App review rating |
| entry_date | date | Date of entry |
| thumbs_up | int | Play Store helpful votes |
| upvotes | int | Reddit upvotes |
| duration_mins | str | Class duration |
| calories_burn | str | Estimated calories |
| difficulty | str | Easy/Medium/Hard |